In [1]:
import pandas as pd
import numpy as np

In [2]:
balls = pd.read_parquet("../ml-service/data/processed/v4_beta/clean_deliveries.parquet")
matches = pd.read_parquet("../ml-service/data/processed/v4_beta/clean_matches.parquet")

In [3]:
balls.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,Sourav Ganguly,Brendon McCullum,Praveen Kumar,0,0,0,0.0,1.0,0.0,Not Out,2008-04-18,1.0,1
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0,2
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,1,0,0.0,0.0,0.0,Not Out,2008-04-18,1.0,3
3,335982,0,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0,4
4,335982,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0,5


In [4]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'total_balls'],
      dtype='object')

In [5]:
balls.groupby(["matchId", "inning", "over", "total_balls"]).size().value_counts()

1    273853
Name: count, dtype: int64

In [6]:
balls.shape

(273853, 19)

In [7]:
balls = balls.sort_values(["matchId", "inning", "over", "total_balls"]).reset_index(
    drop=True
)

In [8]:
balls.groupby(["matchId", "inning"])["matchId"].nunique().value_counts()

matchId
1    2284
Name: count, dtype: int64

In [9]:
mask = balls["isNoBall"] > 1

balls.loc[mask, "batsman_runs"] += balls.loc[mask, "isNoBall"] - 1
balls.loc[mask, "isNoBall"] = 1

In [10]:
balls["total_balls"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [11]:
balls["repeat"] = np.where(balls["isWide"] > 0, balls["isWide"], 1)

balls = balls.loc[balls.index.repeat(balls["repeat"])].copy()

balls.loc[balls["isWide"] > 0, "isWide"] = 1

balls.drop(columns=["repeat"], inplace=True)

In [12]:
balls.shape

(275637, 19)

In [13]:
balls["is_legal"] = ((balls["isWide"] == 0) & (balls["isNoBall"] == 0)).astype(int)

balls["ball"] = balls.groupby(["matchId", "inning", "over"])["is_legal"].cumsum()

balls["ball"] = balls["ball"].replace(0, np.nan)
balls["ball"] = balls.groupby(["matchId", "inning", "over"])["ball"].ffill().fillna(1)

In [14]:
balls = balls.reset_index(drop=True)

In [15]:
balls.shape

(275637, 20)

In [16]:
balls = balls[balls["ball"] <= 6]

In [17]:
balls.shape

(275633, 20)

In [18]:
balls = balls.rename(columns={"ball": "legal_ball"})

In [19]:
balls["legal_ball"].unique()

array([1., 2., 3., 4., 5., 6.])

In [20]:
balls.groupby(["matchId", "inning", "over"])["is_legal"].sum().max()

np.int64(6)

In [21]:
balls["legal_ball_1"] = (balls["isWide"] == 0) & (balls["isNoBall"] == 0)

balls["balls_bowled"] = balls.groupby(["matchId", "inning"])["legal_ball_1"].cumsum()

balls["balls_remaining"] = 120 - balls["balls_bowled"]

balls.drop(columns=["balls_bowled"], inplace=True)

In [22]:
balls.groupby(["matchId", "inning"])["legal_ball_1"].sum().max()

np.int64(120)

In [23]:
balls = balls.reset_index(drop=True)

In [24]:
balls["balls_remaining"].unique()

array([119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108, 107,
       106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,  94,
        93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,  81,
        80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,  68,
        67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,  55,
        54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,  42,
        41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,  29,
        28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,  16,
        15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,   3,
         2,   1,   0, 120])

In [25]:
balls["over_number"] = balls["over"].astype(int) + 1
balls["phase"] = np.select(
    [balls["over_number"] <= 6, balls["over_number"] <= 15], [0, 1], default=2
)
# balls.drop(columns=['over_number'], inplace=True)

In [26]:
balls["total_runs"] = (
    balls["batsman_runs"]
    + balls["isWide"]
    + balls["isNoBall"]
    + balls["Byes"]
    + balls["LegByes"]
    + balls["Penalty"]
)
balls["current_score"] = balls.groupby(["matchId", "inning"])["total_runs"].cumsum()
# balls.drop(columns=['total_runs'], inplace=True)

In [27]:
balls[["over", "legal_ball", "current_score", "isWide", "isNoBall"]].head(10)

,over,legal_ball,current_score,isWide,isNoBall
0,0,1.0,1.0,0,0
1,0,2.0,1.0,0,0
2,0,2.0,2.0,1,0
3,0,3.0,2.0,0,0
4,0,4.0,2.0,0,0
5,0,5.0,2.0,0,0
6,0,6.0,3.0,0,0
7,1,1.0,3.0,0,0
8,1,2.0,7.0,0,0
9,1,3.0,11.0,0,0


In [28]:
balls["current_score"].unique().max()

np.float64(287.0)

In [29]:
balls.loc[balls["Penalty"] == 5, "batsman_runs"] = 5

In [30]:
balls["batsman_runs"] = balls["batsman_runs"] + balls["Byes"] + balls["LegByes"]
balls["batsman_runs"].unique()

array([1., 0., 4., 6., 2., 5., 3.])

In [31]:
balls = balls.reset_index(drop=True)

In [32]:
len(balls["player_dismissed"].unique())

648

In [33]:
balls["is_wicket"] = (balls["player_dismissed"] != "Not Out").astype(int)
balls["wickets_fallen"] = balls.groupby(["matchId", "inning"])["is_wicket"].cumsum()

In [34]:
balls["wickets_fallen"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [35]:
balls = balls.reset_index(drop=True)

In [36]:
first_innings_score = (
    balls[balls["inning"] == 0].groupby("matchId")["current_score"].max()
)
balls["target"] = balls["matchId"].map(first_innings_score)
balls.loc[balls["inning"] == 1, "target"] = balls["target"] + 1
balls.loc[balls["inning"] == 0, "target"] = 0

In [37]:
balls[balls["target"] > 0].head(2)

,matchId,inning,over,legal_ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,total_balls,is_legal,legal_ball_1,balls_remaining,over_number,phase,current_score,is_wicket,wickets_fallen,target
129,335982,1,0,1.0,Royal Challengers Bangalore,Kolkata Knight Riders,Rahul Dravid,Wasim Jaffer,Ashok Dinda,1.0,...,1,1,True,119,1,0,1.0,0,0,223.0
130,335982,1,0,1.0,Royal Challengers Bangalore,Kolkata Knight Riders,Wasim Jaffer,Rahul Dravid,Ashok Dinda,0.0,...,2,0,False,119,1,0,2.0,0,0,223.0


In [38]:
balls["target"].unique().max()

np.float64(288.0)

In [39]:
over_runs = (
    balls.groupby(["matchId", "inning", "over_number"])["total_runs"]
    .sum()
    .reset_index(name="over_runs")
)
over_runs["last_over_runs"] = over_runs.groupby(["matchId", "inning"])[
    "over_runs"
].shift(1)
balls = balls.merge(
    over_runs[["matchId", "inning", "over_number", "last_over_runs"]],
    on=["matchId", "inning", "over_number"],
    how="left",
)
balls["last_over_runs"] = balls["last_over_runs"].fillna(0)
balls["last_over_runs"] = balls["last_over_runs"].astype(int)

In [40]:
balls["total_balls"] = balls.groupby(["matchId", "inning", "over"]).cumcount() + 1

In [41]:
balls["total_balls"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [42]:
balls.groupby(
    ["matchId", "inning", "over", "legal_ball", "total_balls"]
).size().value_counts()

1    275633
Name: count, dtype: int64

In [43]:
balls[(balls["matchId"] == 1473467) & (balls["inning"] == 0) & (balls["over"] == 19)]

,matchId,inning,over,legal_ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,is_legal,legal_ball_1,balls_remaining,over_number,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs
265932,1473467,0,19,1.0,Lucknow Super Giants,Chennai Super Kings,Rishabh Pant,Abdul Samad,Matheesha Pathirana,0.0,...,0,False,6,20,2,156.0,0,4,0.0,16
265933,1473467,0,19,1.0,Lucknow Super Giants,Chennai Super Kings,Rishabh Pant,Abdul Samad,Matheesha Pathirana,1.0,...,1,True,5,20,2,157.0,0,4,0.0,16
265934,1473467,0,19,1.0,Lucknow Super Giants,Chennai Super Kings,Abdul Samad,Rishabh Pant,Matheesha Pathirana,0.0,...,0,False,5,20,2,158.0,1,5,0.0,16
265935,1473467,0,19,2.0,Lucknow Super Giants,Chennai Super Kings,Rishabh Pant,David Miller,Matheesha Pathirana,0.0,...,1,True,4,20,2,158.0,1,6,0.0,16
265936,1473467,0,19,2.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,0.0,...,0,False,4,20,2,159.0,0,6,0.0,16
265937,1473467,0,19,3.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,0.0,...,1,True,3,20,2,159.0,0,6,0.0,16
265938,1473467,0,19,3.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,0.0,...,0,False,3,20,2,160.0,0,6,0.0,16
265939,1473467,0,19,4.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,2.0,...,1,True,2,20,2,162.0,0,6,0.0,16
265940,1473467,0,19,5.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,4.0,...,1,True,1,20,2,166.0,0,6,0.0,16
265941,1473467,0,19,6.0,Lucknow Super Giants,Chennai Super Kings,Shardul Thakur,David Miller,Matheesha Pathirana,0.0,...,1,True,0,20,2,166.0,1,7,0.0,16


In [44]:
balls.shape

(275633, 29)

In [45]:
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 4
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "current_score",
] = 181
to_drop = balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [46]:
balls.shape

(275630, 29)

In [47]:
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 2
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "current_score",
] = 111
to_drop = balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [48]:
balls.shape

(275628, 29)

In [49]:
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "batsman_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "total_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "current_score",
] = 131
to_drop = balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] > 4)
].index

balls = balls.drop(to_drop)

In [50]:
balls.shape

(275627, 29)

In [51]:
balls = balls.sort_values(["matchId", "inning", "over", "total_balls"]).reset_index(
    drop=True
)

In [52]:
balls["is_boundary"] = balls["batsman_runs"].isin([4, 6]).astype(int)


def compute_balls_since_boundary(x):
    groups = x.cumsum()
    result = x.groupby(groups).cumcount()
    result[groups == 0] = range((groups == 0).sum())

    return result


balls["balls_since_boundary"] = balls.groupby(["matchId", "inning"])[
    "is_boundary"
].transform(compute_balls_since_boundary)

In [53]:
balls["balls_since_boundary"].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100])

In [54]:
len(
    balls[
        (balls["balls_since_boundary"] > 1)
        & (balls["over"] == 0)
        & (balls["total_balls"] == 1)
    ]
)

0

In [55]:
balls["percentage_target_achieved"] = np.where(
    balls["inning"] == 0, 0.0, balls["current_score"] / balls["target"]
)
balls["percentage_target_achieved"] = (
    balls["percentage_target_achieved"].replace([np.inf, -np.inf], 0).fillna(0)
)

In [56]:
balls["percentage_target_achieved"][balls["inning"] == 1].head(2)

129    0.004484
130    0.008969
Name: percentage_target_achieved, dtype: float64

In [57]:
balls = balls.merge(matches[["matchId", "venue"]], on="matchId", how="left")

In [58]:
balls["venue"].head(240)

0      Chinnaswamy Stadium, Bengaluru
1      Chinnaswamy Stadium, Bengaluru
2      Chinnaswamy Stadium, Bengaluru
3      Chinnaswamy Stadium, Bengaluru
4      Chinnaswamy Stadium, Bengaluru
                    ...              
235         IS Bindra Stadium, Mohali
236         IS Bindra Stadium, Mohali
237         IS Bindra Stadium, Mohali
238         IS Bindra Stadium, Mohali
239         IS Bindra Stadium, Mohali
Name: venue, Length: 240, dtype: object

In [59]:
balls["balls_bowled"] = balls.groupby(["matchId", "inning"])["legal_ball_1"].cumsum()

balls["overs_bowled"] = balls["balls_bowled"] / 6

balls["current_run_rate"] = np.where(
    balls["balls_bowled"] > 0, balls["current_score"] / balls["overs_bowled"], 0
)

In [60]:
balls["runs_required"] = balls["target"] - balls["current_score"]

balls["overs_remaining"] = balls["balls_remaining"] / 6

balls["required_run_rate"] = balls["runs_required"] / balls["overs_remaining"]

# handle divide by 0
balls["required_run_rate"] = balls["required_run_rate"].replace([np.inf, -np.inf], 0)
balls["required_run_rate"] = balls["required_run_rate"].fillna(0)

balls.loc[balls["inning"] == 0, "required_run_rate"] = 0
balls.loc[balls["runs_required"] <= 0, "required_run_rate"] = 0

In [61]:
balls[["current_run_rate", "required_run_rate"]].describe()

,current_run_rate,required_run_rate
count,275627.000000,275627.000000
mean,7.659132,5.241400
std,2.430781,10.791865
min,0.000000,0.000000
25%,6.396226,0.000000
50%,7.655172,0.000000
75%,8.923077,8.961039
max,66.000000,792.000000


In [62]:
pd.set_option("display.max_columns", None)
balls[balls["current_run_rate"] > 42].head()

,matchId,inning,over,legal_ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls,is_legal,legal_ball_1,balls_remaining,over_number,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,is_boundary,balls_since_boundary,percentage_target_achieved,venue,balls_bowled,overs_bowled,current_run_rate,runs_required,overs_remaining,required_run_rate
46915,501222,0,0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,Jacques Kallis,Brad Haddin,Zaheer Khan,0.0,1,0,0.0,0.0,0.0,Not Out,2011-04-22,1.0,5,0,False,119,1,0,8.0,0,0,0.0,0,0,4,0.000000,"Eden Gardens, Kolkata",1,0.166667,48.0,-8.0,19.833333,0.000000
46916,501222,0,0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,Jacques Kallis,Brad Haddin,Zaheer Khan,0.0,1,0,0.0,0.0,0.0,Not Out,2011-04-22,1.0,6,0,False,119,1,0,9.0,0,0,0.0,0,0,5,0.000000,"Eden Gardens, Kolkata",1,0.166667,54.0,-9.0,19.833333,0.000000
46917,501222,0,0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,Jacques Kallis,Brad Haddin,Zaheer Khan,0.0,1,0,0.0,0.0,0.0,Not Out,2011-04-22,1.0,7,0,False,119,1,0,10.0,0,0,0.0,0,0,6,0.000000,"Eden Gardens, Kolkata",1,0.166667,60.0,-10.0,19.833333,0.000000
46918,501222,0,0,1.0,Kolkata Knight Riders,Royal Challengers Bangalore,Jacques Kallis,Brad Haddin,Zaheer Khan,0.0,1,0,0.0,0.0,0.0,Not Out,2011-04-22,1.0,8,0,False,119,1,0,11.0,0,0,0.0,0,0,7,0.000000,"Eden Gardens, Kolkata",1,0.166667,66.0,-11.0,19.833333,0.000000
84883,598034,1,0,1.0,Kolkata Knight Riders,Chennai Super Kings,Manvinder Bisla,Gautam Gambhir,Dirk Nannes,4.0,0,0,4.0,0.0,0.0,Not Out,2013-04-28,4.0,7,1,True,119,1,0,10.0,0,0,201.0,0,1,0,0.049751,"Chepauk Stadium, Chennai",1,0.166667,60.0,191.0,19.833333,9.630252


In [63]:
key_cols = ["matchId", "inning", "over", "total_balls"]

multi_batters = (
    balls.groupby(key_cols)["batsman"].nunique().reset_index(name="batter_count")
)

bad_balls_batter = multi_batters[multi_batters["batter_count"] > 1]

print(bad_balls_batter.shape)
bad_balls_batter.head()

(0, 5)


,matchId,inning,over,total_balls,batter_count


In [64]:
multi_bowlers = (
    balls.groupby(key_cols)["bowler"].nunique().reset_index(name="bowler_count")
)

bad_balls_bowler = multi_bowlers[multi_bowlers["bowler_count"] > 1]

print(bad_balls_bowler.shape)
bad_balls_bowler.head()

(0, 5)


,matchId,inning,over,total_balls,bowler_count


In [65]:
balls.index.is_unique

True

In [66]:
mask = (balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")
balls.loc[mask, "isWide"] = 1
balls.loc[mask, "isNoBall"] = 0
len(balls[(balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")])

0

In [67]:
len(balls[(balls["isWide"] == 1) & (balls["player_dismissed"] != "Not Out")])

61

In [68]:
balls["batsman_runs_target"] = balls["batsman_runs"].astype(int)
balls["isWide_target"] = balls["isWide"].astype(int)
balls["isNoBall_target"] = balls["isNoBall"].astype(int)
balls["is_wicket_target"] = balls["is_wicket"].astype(int)

In [69]:
balls["over"] = balls["over"] / 20
np.set_printoptions(suppress=True)
balls["over"].unique()

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [70]:
season_map = matches.set_index("matchId")["season"]
balls["season"] = balls["matchId"].map(season_map)

In [71]:
# create cyclic features
balls["sin_ball"] = np.sin(2 * np.pi * balls["legal_ball"] / 6)
balls["cos_ball"] = np.cos(2 * np.pi * balls["legal_ball"] / 6)

In [72]:
balls.columns

Index(['matchId', 'inning', 'over', 'legal_ball', 'batting_team',
       'bowling_team', 'batsman', 'non_striker', 'bowler', 'batsman_runs',
       'isWide', 'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed',
       'date', 'total_runs', 'total_balls', 'is_legal', 'legal_ball_1',
       'balls_remaining', 'over_number', 'phase', 'current_score', 'is_wicket',
       'wickets_fallen', 'target', 'last_over_runs', 'is_boundary',
       'balls_since_boundary', 'percentage_target_achieved', 'venue',
       'balls_bowled', 'overs_bowled', 'current_run_rate', 'runs_required',
       'overs_remaining', 'required_run_rate', 'batsman_runs_target',
       'isWide_target', 'isNoBall_target', 'is_wicket_target', 'season',
       'sin_ball', 'cos_ball'],
      dtype='object')

In [73]:
balls.drop(
    columns=[
        "legal_ball",
        "batting_team",
        "bowling_team",
        "Byes",
        "LegByes",
        "Penalty",
        "over_number",
        "is_legal",
        "legal_ball_1",
        "is_boundary",
        "legal_ball",
        "balls_bowled",
        "overs_bowled",
        "runs_required",
        "overs_remaining",
        "date",
    ],
    inplace=True,
)
balls.head()

,matchId,inning,over,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,player_dismissed,total_runs,total_balls,balls_remaining,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,batsman_runs_target,isWide_target,isNoBall_target,is_wicket_target,season,sin_ball,cos_ball
0,335982,0,0.0,Sourav Ganguly,Brendon McCullum,Praveen Kumar,1.0,0,0,Not Out,1.0,1,119,0,1.0,0,0,0.0,0,0,0.0,"Chinnaswamy Stadium, Bengaluru",6.0,0.0,1,0,0,0,2008,8.660254e-01,0.5
1,335982,0,0.0,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.0,0,0,Not Out,0.0,2,118,0,1.0,0,0,0.0,0,1,0.0,"Chinnaswamy Stadium, Bengaluru",3.0,0.0,0,0,0,0,2008,8.660254e-01,-0.5
2,335982,0,0.0,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.0,1,0,Not Out,1.0,3,118,0,2.0,0,0,0.0,0,2,0.0,"Chinnaswamy Stadium, Bengaluru",6.0,0.0,0,1,0,0,2008,8.660254e-01,-0.5
3,335982,0,0.0,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.0,0,0,Not Out,0.0,4,117,0,2.0,0,0,0.0,0,3,0.0,"Chinnaswamy Stadium, Bengaluru",4.0,0.0,0,0,0,0,2008,1.224647e-16,-1.0
4,335982,0,0.0,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.0,0,0,Not Out,0.0,5,116,0,2.0,0,0,0.0,0,4,0.0,"Chinnaswamy Stadium, Bengaluru",3.0,0.0,0,0,0,0,2008,-8.660254e-01,-0.5


In [74]:
balls.columns

Index(['matchId', 'inning', 'over', 'batsman', 'non_striker', 'bowler',
       'batsman_runs', 'isWide', 'isNoBall', 'player_dismissed', 'total_runs',
       'total_balls', 'balls_remaining', 'phase', 'current_score', 'is_wicket',
       'wickets_fallen', 'target', 'last_over_runs', 'balls_since_boundary',
       'percentage_target_achieved', 'venue', 'current_run_rate',
       'required_run_rate', 'batsman_runs_target', 'isWide_target',
       'isNoBall_target', 'is_wicket_target', 'season', 'sin_ball',
       'cos_ball'],
      dtype='object')

In [75]:
balls["batsman_runs"] = balls["batsman_runs"] / 6
balls["total_runs"] = balls["total_runs"] / 6
balls["balls_remaining"] = balls["balls_remaining"] / 120
balls["wickets_fallen"] = balls["wickets_fallen"] / 10
balls["balls_since_boundary"] = balls["balls_since_boundary"] / 120

In [76]:
balls["current_score"] = balls["current_score"] / 200
balls["target"] = balls["target"] / 200
balls["last_over_runs"] = balls["last_over_runs"] / 200
balls["total_balls"] = balls["total_balls"] / 10

In [77]:
balls["season"] = (balls["season"] - 2007) / 20

In [78]:
balls["current_run_rate"] = balls["current_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"].clip(upper=2)

In [79]:
balls["required_run_rate"].unique().max()

np.float64(2.0)

In [80]:
balls[balls["percentage_target_achieved"] >= 1.05]

,matchId,inning,over,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,player_dismissed,total_runs,total_balls,balls_remaining,phase,current_score,is_wicket,wickets_fallen,target,last_over_runs,balls_since_boundary,percentage_target_achieved,venue,current_run_rate,required_run_rate,batsman_runs_target,isWide_target,isNoBall_target,is_wicket_target,season,sin_ball,cos_ball
147981,1082645,1,0.55,Ajinkya Rahane,Steve Smith,Glenn Maxwell,1.0,0,0,Not Out,1.0,0.6,0.400000,1,0.390,0,0.1,0.37,0.025,0.0,1.054054,"MCA Stadium, Pune",0.180556,0.0,6,0,0,0,0.50,-2.449294e-16,1.0
220433,1304105,1,0.70,Tim David,Tilak Varma,Moeen Ali,1.0,0,0,Not Out,1.0,0.5,0.258333,1,0.515,0,0.5,0.49,0.030,0.0,1.051020,"Wankhede Stadium, Mumbai",0.192884,0.0,6,0,0,0,0.75,-8.660254e-01,0.5


In [81]:
# Verify target distributions
print("batsman_runs_target:", balls["batsman_runs_target"].value_counts().sort_index())
print("isWide_target:", balls["isWide_target"].value_counts())
print("isNoBall_target:", balls["isNoBall_target"].value_counts())
print("is_wicket_target:", balls["is_wicket_target"].value_counts())

# Verify no NaN in targets
print(
    "NaN count:",
    balls[
        ["batsman_runs_target", "isWide_target", "isNoBall_target", "is_wicket_target"]
    ]
    .isna()
    .sum(),
)

batsman_runs_target: batsman_runs_target
0    105481
1    105665
2     17377
3       823
4     32115
5        73
6     14093
Name: count, dtype: int64
isWide_target: isWide_target
0    264897
1     10730
Name: count, dtype: int64
isNoBall_target: isNoBall_target
0    274501
1      1126
Name: count, dtype: int64
is_wicket_target: is_wicket_target
0    262046
1     13581
Name: count, dtype: int64
NaN count: batsman_runs_target    0
isWide_target          0
isNoBall_target        0
is_wicket_target       0
dtype: int64


In [82]:
balls.to_csv("data2.csv")